# Exploratory notebook

Kept as run during the competition, with one change: the original absolute
paths on Stanford's Sherlock cluster have been replaced by `eegchallenge.config`
and the `EEGCHALLENGE_*` environment variables, as everywhere else in the
repository. `source config/paths.sh` before running.

The directories these cells originally read no longer exist (`$SCRATCH` was
purged), so the cells will not reproduce without regenerating the summaries
first. See the README section "Reproducibility notes".


In [ ]:
%config InlineBackend.figure_formats = {'svg',}


import sys
from pathlib import Path
from itertools import product
import pandas as pd
import numpy as np
from scipy import signal
import torch
import seaborn as sns
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from sklearn.linear_model import SGDRegressor,RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import FunctionTransformer,StandardScaler
from tqdm import tqdm

from eegchallenge import config
from eegchallenge.features import log10_safe
# NB: this notebook originally called load_train_test(), which returned
# (X_PSD_train, y_train, X_PSD_test, y_test). That function was removed from
# scripts/challenges/challenge2/supervised_linear.py before this release and
# has no direct equivalent. The nearest surviving function is
# eegchallenge.pipeline.load_X_y_test_fold(challenge), which instead returns
# (X, y, test_fold), where test_fold marks each row as training (-1) or model
# selection (0). The RECOMPUTE cells below would need rewriting against it; the
# saved .csv summaries are what the plots actually read.


RECOMPUTE = False


TO = config.work_root() / 'distribution_shift_problem'
TO.mkdir(parents=True, exist_ok=True)

In [ ]:
def describe(*arrays):
    for array,name in arrays:
        pd.DataFrame(array).describe().to_csv(TO/f'{name}.csv')


if RECOMPUTE:
    X_PSD_train, y_train, X_PSD_test, y_test = load_train_test()
    describe((X_PSD_train,'X_PSD_train'), (y_train,'y_train'), (X_PSD_test,'X_PSD_test'), (y_test,'y_test'))

# Train verus test distributions for features (PSD) and targets look very similar

## Features

In [ ]:
def displot(train, test, log_scale=False, cumulative=False, common_bins=True):
    train,test = pd.read_csv(TO/f'{train}.csv',index_col=0),pd.read_csv(TO/f'{test}.csv',index_col=0)
    assert (train.index==['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']).all()
    assert (test.index==['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']).all()
    sns.displot(
        pd.concat((train, test), axis=1, keys=['train','test']).T.drop(columns=['count']).reset_index().rename({'level_0':'train_test'}, axis=1).drop(columns=['level_1']).melt(id_vars=['train_test']),
        x='value', col='variable', hue='train_test',
        stat='percent', common_norm=False,
        col_wrap=4, facet_kws={'sharex':False,'sharey':False}, element='step', log_scale=log_scale,
        cumulative=cumulative,
        common_bins=common_bins, # Annoying: want to set to False so that bins are not common across axes, but want to set to True so that bins are same within axes...
        bins=100
    )


def compute_overlap(train, test):
    train = pd.read_csv(TO/f'{train}.csv',index_col=0)
    test = pd.read_csv(TO/f'{test}.csv',index_col=0)
    print('Fraction train-test IQRs non-overlapping:',
        (train.loc['75%'] < test.loc['25%']).sum()/train.shape[1] + (test.loc['75%'] < train.loc['25%']).sum()/train.shape[1])


displot('X_PSD_train','X_PSD_test')
displot('X_PSD_train','X_PSD_test',cumulative=True,common_bins=False)
compute_overlap('X_PSD_train','X_PSD_test')

## Targets

In [ ]:
pd.read_csv(TO/f'y_train.csv',index_col=0)

In [ ]:
pd.read_csv(TO/f'y_test.csv',index_col=0)

# log10_safe

Observations
- Data are extremely skewed low (because observations that are 0 get mapped to -300)
- Train data have min <= test data (because more subjects in train data mean more likely for a feature to be 0)

Questions
- (I think) because IQRs are extremely tight, most test and train IQRs are fully non-overlapping! Is this a problem for generalization?

In [ ]:
def fit_transform_describe(pipe, X, name):
    describe((pipe.fit_transform(X), name))


if RECOMPUTE:
    pipe_log10_safe = make_pipeline(
        VarianceThreshold(),
        FunctionTransformer(log10_safe),
        )
    pipe_log1p = make_pipeline(
        VarianceThreshold(),
        FunctionTransformer(np.log1p),
    )
    pipe_log10_safe_standardize = make_pipeline(
        VarianceThreshold(),
        FunctionTransformer(log10_safe),
        StandardScaler(),
    )
    pipe_log1p_standardize = make_pipeline(
        VarianceThreshold(),
        FunctionTransformer(np.log1p),
        StandardScaler(),
    )
    pipes = {
        'pipe_log10_safe':pipe_log10_safe,
        'pipe_log1p':pipe_log1p,
        'pipe_log10_safe_standardize':pipe_log10_safe_standardize,
        'pipe_log1p_standardize':pipe_log1p_standardize
    }
    Xs = {
        'X_PSD_train':X_PSD_train,
        'X_PSD_test':X_PSD_test,
    }
    for pipe, X in product(pipes, Xs):
        fit_transform_describe(pipes[pipe], Xs[X], f'{pipe}_{X}')

## Just log10_safe

In [ ]:
displot('pipe_log10_safe_X_PSD_train', 'pipe_log10_safe_X_PSD_test')
displot('pipe_log10_safe_X_PSD_train', 'pipe_log10_safe_X_PSD_test', cumulative=True, common_bins=False)
compute_overlap('pipe_log10_safe_X_PSD_train','pipe_log10_safe_X_PSD_test')

## log10_safe then standardize

In [ ]:
displot('pipe_log10_safe_standardize_X_PSD_train', 'pipe_log10_safe_standardize_X_PSD_test',)
displot('pipe_log10_safe_standardize_X_PSD_train', 'pipe_log10_safe_standardize_X_PSD_test', cumulative=True, common_bins=False)
compute_overlap('pipe_log10_safe_standardize_X_PSD_train','pipe_log10_safe_standardize_X_PSD_test')

# log1p

Observations
- There are some masssive outliers (see standardized features)
- Train data has higher max and lower min than test (because more subjects per feature)

Questions
- Because (I think) IQRs are extremely tight, most test and train IQRs are fully non-overlapping! Is this a problem for generalization?

## Just log1p

In [ ]:
displot('pipe_log1p_X_PSD_train', 'pipe_log1p_X_PSD_test', )
displot('pipe_log1p_X_PSD_train', 'pipe_log1p_X_PSD_test', cumulative=True, common_bins=False)
compute_overlap('pipe_log1p_X_PSD_train','pipe_log1p_X_PSD_test')

## log1p then standardize

In [ ]:
displot('pipe_log1p_standardize_X_PSD_train', 'pipe_log1p_standardize_X_PSD_test', )
displot('pipe_log1p_standardize_X_PSD_train', 'pipe_log1p_standardize_X_PSD_test', cumulative=True, common_bins=False)
compute_overlap('pipe_log1p_standardize_X_PSD_train','pipe_log1p_standardize_X_PSD_test')

In [ ]:
_train = pd.read_csv(TO/'pipe_log1p_standardize_X_PSD_train.csv',index_col=0)
_train

In [ ]:
_test = pd.read_csv(TO/'pipe_log1p_standardize_X_PSD_test.csv',index_col=0)
_test